# 🧠 NeuroSight AI — Détection de Pathologies d'Alzheimer
**Version 3 — Corrigée (Split 3 phases · Focal Loss · Early Stopping · Macro-F1)**

| Cellule | Objectif |
|---------|----------|
| 1 | Import dynamique du pipeline depuis GitHub |
| 2 | Chargement des données & création des DataLoaders |
| 3 | Construction de l'architecture EfficientNet-B0 + Focal Loss |
| 4 | Entraînement Phase 1 (Warm-Up tête uniquement) |
| 5 | Entraînement Phase 2 (Fine-Tuning Discriminatif) |
| 6 | Évaluation complète (Classification Report, Confusion Matrix, ROC-AUC) |


In [ ]:
# ==========================================================
# CELLULE 1 : IMPORT DYNAMIQUE DU PIPELINE (depuis GitHub)
# Objectif : Télécharger data_pipeline.py v3 depuis la branche
# dev-wilfried et l'importer dynamiquement dans le notebook.
# ==========================================================
import os
import sys
import importlib

GITHUB_RAW_URL = "https://raw.githubusercontent.com/wekt2k04/NeuroSight_AI/dev-wilfried/notebooks/data_pipeline.py"

print("📥 Mise à jour du pipeline NeuroSight (v3)...")
!wget -q -O data_pipeline.py {GITHUB_RAW_URL}

if '/kaggle/working' not in sys.path:
    sys.path.append('/kaggle/working')

try:
    import data_pipeline
    importlib.reload(data_pipeline)
    from data_pipeline import prepare_data, show_sample_batch, plot_class_distribution
    print("✅ Pipeline v3 chargé avec succès.")
except Exception as e:
    print(f"❌ Erreur d'import : {e}")


In [ ]:
# ==========================================================
# CELLULE 2 : CHARGEMENT DES DONNÉES & CRÉATION DES DATALOADERS
# Objectif : Préparer les DataLoaders train/val via le pipeline
# v3 (split 3 phases — ModerateDemented garanti dans val).
# ==========================================================

CSV_PATH    = '/kaggle/input/datasets/wilfriedtsetse04/mon-csv-officiel/oasis_cross-sectional-5708aa0a98d82080.xlsx'
IMAGES_DIR  = '/kaggle/input/datasets/ninadaithal/imagesoasis/Data'
BATCH_SIZE  = 32

train_loader, val_loader, classes = prepare_data(
    csv_path=CSV_PATH,
    images_dir=IMAGES_DIR,
    batch_size=BATCH_SIZE
)

print(f"\n🚀 Classes : {classes}")

# --- AUDIT VISUEL (décommenter pour explorer les données) ---
# plot_class_distribution(train_loader, classes)
# show_sample_batch(train_loader, classes)


In [ ]:
# ==========================================================
# CELLULE 3 : ARCHITECTURE EFFICIENTNET-B0 + FOCAL LOSS
# Objectif : Définir le modèle, la Focal Loss (adaptée aux
# classes déséquilibrées) et la boucle d'entraînement avec
# Early Stopping sur le Macro F1-Score (pas l'accuracy).
# ==========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import copy
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Moteur d'entraînement amorcé sur : {device}")

# ----------------------------------------------------------
# 1. FOCAL LOSS — Pénalise davantage les erreurs sur classes rares
# alpha : poids inversement proportionnel à la fréquence des classes
#   [NonDemented=1, VeryMild=3, Mild=5, Moderate=10]
# gamma : paramètre de focus (2.0 est la valeur standard)
# ----------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# ----------------------------------------------------------
# 2. CONSTRUCTION DE L'ARCHITECTURE (EFFICIENTNET-B0)
# ----------------------------------------------------------
def build_neurosight_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(num_ftrs, num_classes)
    )
    return model.to(device)

model = build_neurosight_model(len(classes))
print(f"✅ Modèle EfficientNet-B0 construit ({len(classes)} classes).")

# ----------------------------------------------------------
# 3. BOUCLE D'ENTRAÎNEMENT AVEC EARLY STOPPING SUR MACRO F1
# ----------------------------------------------------------
def train_model(model, train_loader, val_loader, num_epochs,
                optimizer, scheduler, phase_name="",
                patience=5, focal_alpha=None):
    """
    Entraîne le modèle avec :
    - Focal Loss pondérée par classe
    - Early Stopping sur le Macro F1-Score de validation
    - Sauvegarde des meilleurs poids selon le Macro F1 (pas l'accuracy)
    """
    criterion = FocalLoss(alpha=focal_alpha, gamma=2.0)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_macro_f1': []}
    best_model_wts = copy.deepcopy(model.state_dict())
    best_macro_f1  = 0.0
    patience_counter = 0

    print(f"\n🚀 Lancement : {phase_name} (max {num_epochs} époques | patience={patience})")
    print("-" * 60)

    for epoch in range(num_epochs):
        start_time = time.time()

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss     = 0.0
            running_corrects = 0
            all_preds, all_labels = [], []

            for inputs, labels in dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc  = running_corrects.double() / len(dataloader.dataset)
            epoch_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                train_loss_val = epoch_loss
                train_acc_val  = epoch_acc
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                history['val_macro_f1'].append(epoch_f1)
                scheduler.step(epoch_loss)

                # Early Stopping & sauvegarde sur Macro F1
                if epoch_f1 > best_macro_f1:
                    best_macro_f1    = epoch_f1
                    best_model_wts   = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                    flag = "💾 (sauvegardé)"
                else:
                    patience_counter += 1
                    flag = f"⏳ patience {patience_counter}/{patience}"

        time_elapsed = time.time() - start_time
        print(f"Époque {epoch+1:02d}/{num_epochs:02d} | {time_elapsed:.0f}s | "
              f"Train Loss: {train_loss_val:.4f} Acc: {train_acc_val:.4f} | "
              f"Val Loss: {history['val_loss'][-1]:.4f} Acc: {history['val_acc'][-1]:.4f} "
              f"F1: {history['val_macro_f1'][-1]:.4f} {flag}")

        if patience_counter >= patience:
            print(f"🛑 Early Stopping déclenché à l'époque {epoch+1}.")
            break

    print(f"✅ Phase terminée. Meilleur Macro F1 de validation : {best_macro_f1:.4f}")
    model.load_state_dict(best_model_wts)
    return model, history


In [ ]:
# ==========================================================
# CELLULE 4 : ENTRAÎNEMENT — PHASE 1 (WARM-UP)
# Objectif : Entraîner uniquement la tête de classification
# pendant 5 époques pour initialiser les poids sans détruire
# les features ImageNet du backbone EfficientNet.
# ==========================================================

# Poids Focal Loss : inversement proportionnels à la fréquence
# [NonDemented=1, VeryMild=3, Mild=5, Moderate=10]
focal_alpha = torch.tensor([1.0, 3.0, 5.0, 10.0], device=device)

# Gel du backbone (feature extractor)
for param in model.features.parameters():
    param.requires_grad = False

optimizer_warmup = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler_warmup = ReduceLROnPlateau(optimizer_warmup, mode='min', factor=0.5, patience=2)

model, hist_warmup = train_model(
    model, train_loader, val_loader,
    num_epochs=5,
    optimizer=optimizer_warmup,
    scheduler=scheduler_warmup,
    phase_name="PHASE 1 : WARM-UP (Tête uniquement)",
    patience=3,
    focal_alpha=focal_alpha
)


In [ ]:
# ==========================================================
# CELLULE 5 : ENTRAÎNEMENT — PHASE 2 (FINE-TUNING DISCRIMINATIF)
# Objectif : Dégeler le réseau entier et appliquer des taux
# d'apprentissage différenciés (backbone très faible, tête
# normale) pour affiner sans catastrophic forgetting.
# Early Stopping sur Macro F1 avec patience=5.
# ==========================================================

# Dégel de tout le réseau
for param in model.parameters():
    param.requires_grad = True

# Discriminative Learning Rates :
# Le backbone s'ajuste très délicatement (5e-6)
# La tête continue d'apprendre normalement (2e-4)
optimizer_ft = optim.AdamW([
    {'params': model.features.parameters(),   'lr': 5e-6},
    {'params': model.classifier.parameters(), 'lr': 2e-4}
], weight_decay=1e-2)

scheduler_ft = ReduceLROnPlateau(optimizer_ft, mode='min', factor=0.5, patience=3)

model, hist_ft = train_model(
    model, train_loader, val_loader,
    num_epochs=20,
    optimizer=optimizer_ft,
    scheduler=scheduler_ft,
    phase_name="PHASE 2 : FINE-TUNING (Discriminative LR)",
    patience=5,
    focal_alpha=focal_alpha
)

# Fusion des historiques pour les graphiques
n_warmup = len(hist_warmup['train_loss'])
full_history = {
    'train_loss':    hist_warmup['train_loss']    + hist_ft['train_loss'],
    'val_loss':      hist_warmup['val_loss']      + hist_ft['val_loss'],
    'val_acc':       hist_warmup['val_acc']       + hist_ft['val_acc'],
    'val_macro_f1':  hist_warmup['val_macro_f1']  + hist_ft['val_macro_f1'],
    'n_warmup_epochs': n_warmup
}
print(f"\n📊 Historique fusionné : {len(full_history['train_loss'])} époques totales.")


In [ ]:
# ==========================================================
# CELLULE 6 : ÉVALUATION COMPLÈTE (RAPPORT + COURBES + ROC-AUC)
# Objectif : Générer le rapport de classification par classe,
# les courbes d'apprentissage (Loss, Accuracy, Macro F1),
# la matrice de confusion et les courbes ROC-AUC multiclasses.
# ==========================================================

def evaluate_and_plot(model, val_loader, class_names, history):
    model.eval()
    y_true, y_pred, y_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    y_probs = np.array(y_probs)
    sep = history.get('n_warmup_epochs', 3) - 0.5

    print("\n" + "="*55)
    print("📊 RAPPORT DE PERFORMANCE MÉDICALE (F1 & RECALL)")
    print("="*55)
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"   🎯 Macro F1-Score global : {macro_f1:.4f}")

    # --- Courbes d'apprentissage (3 panneaux) ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history['train_loss'], label='Train Loss', color='royalblue')
    axes[0].plot(history['val_loss'],   label='Val Loss',   color='darkorange')
    axes[0].axvline(x=sep, color='red', linestyle='--', label='Début Phase 2')
    axes[0].set_title('Trajectoire de la Perte (Focal Loss)')
    axes[0].set_xlabel('Époques'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.5)

    axes[1].plot(history['val_acc'], label='Val Accuracy', color='seagreen')
    axes[1].axvline(x=sep, color='red', linestyle='--', label='Début Phase 2')
    axes[1].set_title('Accuracy de Validation')
    axes[1].set_xlabel('Époques'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.5)

    axes[2].plot(history['val_macro_f1'], label='Val Macro F1', color='darkorchid')
    axes[2].axvline(x=sep, color='red', linestyle='--', label='Début Phase 2')
    axes[2].set_title('Macro F1-Score de Validation ⭐')
    axes[2].set_xlabel('Époques'); axes[2].set_ylabel('Macro F1')
    axes[2].legend(); axes[2].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

    # --- Matrice de Confusion ---
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matrice de Confusion : Diagnostic Alzheimer', fontsize=14, pad=15)
    plt.ylabel('VRAI Diagnostic', fontsize=11)
    plt.xlabel('Diagnostic PRÉDIT', fontsize=11)
    plt.xticks(rotation=45); plt.tight_layout()
    plt.show()

    # --- Courbes ROC-AUC Multiclasse ---
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    plt.figure(figsize=(10, 8))
    for i in range(len(class_names)):
        if np.sum(y_true_bin[:, i]) > 0:
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[i], lw=2,
                     label=f'ROC {class_names[i]} (AUC = {roc_auc:.2f})')
    plt.plot([0,1],[0,1],'k--', lw=2, label='Aléatoire (AUC = 0.50)')
    plt.xlim([-0.05, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('Taux de Faux Positifs (1 - Spécificité)', fontsize=12)
    plt.ylabel('Taux de Vrais Positifs (Sensibilité)', fontsize=12)
    plt.title('Courbes ROC-AUC par Stade de Démence', fontsize=14)
    plt.legend(loc='lower right'); plt.grid(alpha=0.3)
    plt.show()

# Génération des résultats finaux
evaluate_and_plot(model, val_loader, classes, full_history)
